# R28_v2 ProdigyTuning — ricalibrato su R24F_mixed_lr0.5_V08

## Contesto

Tutti i baseline pre-2026-06-12 (R25_B1, R25_A3, R28_A0, R29_E0) avevano gradienti `gn_total_preclip` ∈ [10⁵, 10¹⁷] mascherati dal `clip_grad_norm_(1.0)`. R24F_mixed_lr0.5_V08 è l'UNICO setup post-fix CLEAN (gn_max 21.8). Tutti i v2 partono da quel baseline.

## Setup chiave

- Optimizer: Prodigy `lr=0.5` (NON 1.0)
- Scheduler: cosine_no_restart
- seq_len: 50 (NON 100)
- λ_T_aux: 0.0 nel baseline (varianti testate)
- Explosion guard ATTIVO: `--max_epoch_explosion_streak 2 --epoch_explosion_threshold 100`

## Sweep

5 esperimenti: d0 fix, step budget 3x, warm restart, su lr=0.5 stable.

## Output

`results/Prodigy_Study/R28_ProdigyTuning_v2/<axis>/<tag>/`

In [ ]:
# Cell 1 -- Bootstrap + GLOBALS + ENV check
import sys, os, subprocess
import importlib.util as _imu

RESULTS_DIR = 'results/Prodigy_Study/R28_ProdigyTuning_v2'
AGGREGATE_CSV = f'{RESULTS_DIR}/_aggregate.csv'
BRANCH = 'Prodigy_Deep_Study'
_TMP_MSG = '/tmp/r28v2_msg.txt' if os.path.isdir('/tmp') else 'r28v2_msg.txt'
os.makedirs(RESULTS_DIR, exist_ok=True)

for pkg in ['prodigyopt', 'pandas', 'matplotlib']:
    if _imu.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

for f in ['train.py', 'core/network.py',
          'Arch_Tested/R24F_MIXED_lr0.5_V08_TRUE_CHAMPION/snapshot_original/training_log.csv']:
    assert os.path.isfile(f), f'MISSING: {f}'

sys.path.insert(0, '.')
for mod in ['train','core.network']:
    if mod in sys.modules: del sys.modules[mod]
from train import CSVLogger, BatchCSVLogger, pinn_loss
assert 'val_T_intra_corr' in CSVLogger.COLS
print(f'  [OK] CSVLogger has val_T_intra_corr')

help_txt = subprocess.run([sys.executable, 'train.py', '--help'],
                           capture_output=True, text=True, encoding='utf-8',
                           errors='replace').stdout
for flag in ['--max_epoch_explosion_streak', '--epoch_explosion_threshold',
             '--lambda_v0_aux', '--lambda_s0_aux', '--lambda_a_aux', '--lambda_b_aux',
             '--cf_init_bias_shift', '--cf_logit_tau_init']:
    assert flag in help_txt, f'MISSING CLI: {flag}'
print(f'  [OK] all R29+R30 CLI flags present')

br = subprocess.run(['git','branch','--show-current'], capture_output=True, text=True).stdout.strip()
assert br == BRANCH, f'Branch errato: {br}'
print(f'  [OK] branch={br}')
print(f'\n[R28_v2 ProdigyTuning] ENV check passed.')
print(f'  RESULTS_DIR = {RESULTS_DIR}')

In [ ]:
# Cell 2 -- R28_v2 ProdigyTuning experiments
# === R24F_mixed_lr0.5_V08 BASELINE COMMON CONFIG ===
# Riferimento ufficiale: Arch_Tested/R24F_MIXED_lr0.5_V08_TRUE_CHAMPION/
# CHANGE rispetto a R25_B1 / R29_A3 (vecchi baseline lr=1.0 instabili):
#   lr 1.0 -> 0.5  (gradienti CLEAN gn_max 21.8 vs 10^5-10^17)
#   lambda_T_aux 0.1 -> 0.0  (no T-aux nel baseline, ablation testa varianti)
#   seq_len 100 -> 50  (config R24F originale)
#   AGGIUNTO --max_epoch_explosion_streak 2 + threshold 100 (R30 explosion guard)
R24F_BASELINE = {
    "optimizer": "prodigy", "lr": 0.5,                  # ← CHIAVE: 0.5 NOT 1.0
    "d0": 1e-6, "d_coef": 1.0, "growth_rate": "inf",
    "epochs": 10, "max_steps_per_epoch": 100,
    "seq_len": 50,                                      # ← CHIAVE: 50 NOT 100
    "hidden_size": 32, "rank": 8,
    "max_delay": 6, "bit_shift": 3,
    "lambda_sr": 0.5, "lambda_T_aux": 0.0,              # ← CHIAVE: 0.0 NOT 0.1
    "lambda_v0_aux": 0.0, "lambda_s0_aux": 0.0,
    "lambda_a_aux":  0.0, "lambda_b_aux":  0.0,
    "scenario_mix": "highway:0.4,urban:0.3,truck:0.2,mixed:0.1",
    "cut_in_ratio": 0.0,
    "scheduler": "cosine_no_restart",
    "T0": 5,
    "cache_path": "data/cache_1500_mixed_cut0.0_ou0.0.pt",
    "po2_enabled": 1,
    # R29 decoder fix opt-in (default no-op)
    "init_bias_shift": 0,
    "tau_init": 1.0, "tau_final": 1.0,
    "tau_schedule": "const",
    "tau_per_channel": None,
    # R30 explosion guard ATTIVO di default nei v2
    "max_epoch_explosion_streak": 2,
    "epoch_explosion_threshold":  100.0,
}

def _exp(tag, desc, axis, **overrides):
    e = {**R24F_BASELINE, "tag": tag, "desc": desc, "axis": axis}
    e.update(overrides)
    return e


EXPERIMENTS = [
    _exp('R28v2_A0_baseline_lr05', 'R24F baseline replica (sanity)',
         'A_d0'),  # d0=1e-6 default
    _exp('R28v2_A1_d0_1e-5', 'd0=1e-5 fix konstmish Issue #27',
         'A_d0', d0=1e-5),
    _exp('R28v2_B1_steps_300', 'max_steps=300 (3x budget)',
         'B_steps', max_steps_per_epoch=300),
    _exp('R28v2_C1_d0_steps_combo', 'd0=1e-5 + steps=300',
         'C_combo', d0=1e-5, max_steps_per_epoch=300),
    _exp('R28v2_D1_warm_restart', 'cosine warm restart T0=5 + d0 + steps',
         'D_warm_restart', d0=1e-5, max_steps_per_epoch=300, scheduler='cosine'),
]

print(f'EXPERIMENTS: {len(EXPERIMENTS)}')
for e in EXPERIMENTS:
    print(f"  [{e['axis']:<14}] {e['tag']:<32} -- {e.get('desc','')}")

In [ ]:
# Cell 3 -- Cache check (riuso R24F mixed cache)
import os
cache = R24F_BASELINE['cache_path']
if os.path.isfile(cache):
    sz_mb = os.path.getsize(cache) / 1e6
    print(f'  [OK] cache esistente: {cache}  ({sz_mb:.1f} MB)')
else:
    print(f'  [INFO] cache mancante: {cache} -- verra\' generata dal primo run')

In [ ]:
# Cell 4 -- Pre-flight smoke 1ep × 3step (validare CLI + R24F baseline config)
import torch, time, shutil, sys, subprocess
sys.path.insert(0, '.')
if 'core.network' in sys.modules: del sys.modules['core.network']
from core.network import CF_FSNN_Net

torch.manual_seed(42)
m = CF_FSNN_Net(hidden_size=32, rank=8, max_delay=6, bit_shift=3)
n_params = sum(p.numel() for p in m.parameters())
assert n_params == 864, f'Param count: {n_params}'
print(f'  [OK] CF_FSNN_Net: {n_params} param')

tag_smoke = f'_PREFLIGHT_v2_{int(time.time())}'
cmd_smoke = [sys.executable, 'train.py',
    '--training_method', 'baseline',
    '--epochs', '1', '--max_steps_per_epoch', '3',
    '--batch_size', '8', '--val_batch_size', '32',
    '--seq_len', '50',
    '--cf_hidden_size', '32', '--cf_rank', '8',
    '--cf_max_delay', '6', '--cf_bit_shift', '3',
    '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
    '--lambda_bc', '1.0', '--lambda_sr', '0.5', '--lambda_T_aux', '0.0',
    '--scenario_mix', R24F_BASELINE['scenario_mix'],
    '--cut_in_ratio', '0.0', '--noise_scale', '0.0',
    '--n_train', '80', '--n_val', '40',
    '--optimizer', 'prodigy', '--lr', '0.5',  # R24F LR
    '--scheduler', 'cosine_no_restart',
    '--prodigy_betas', '0.9,0.99', '--prodigy_d_coef', '1.0',
    '--prodigy_d0', '1e-6', '--prodigy_weight_decay', '0.01',
    '--prodigy_use_bias_correction', '1', '--prodigy_safeguard_warmup', '1',
    '--prodigy_growth_rate', 'inf',
    '--max_inf_streak', '99999', '--early_stop_patience', '0',
    '--max_epoch_explosion_streak', '2', '--epoch_explosion_threshold', '100.0',
    '--tag', tag_smoke]
r = subprocess.run(cmd_smoke, capture_output=True, text=True, encoding='utf-8', errors='replace')
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])
    raise RuntimeError(f'preflight failed: {r.returncode}')

import pandas as pd, math
bdf = pd.read_csv(f'checkpoints/{tag_smoke}/training_batch_log.csv')
gn = bdf['gn_total_preclip']
gn_f = gn[gn.apply(math.isfinite)]
print(f'  [OK] Pre-flight gn_max={gn_f.max():.3f} (clean if < 25)')

import os
import shutil as _shutil
def _robust_rmtree(path, max_retries=3):
    for attempt in range(max_retries):
        if not os.path.isdir(path): return True
        _shutil.rmtree(path, ignore_errors=True)
        if not os.path.isdir(path): return True
        time.sleep(0.5 * (attempt + 1))
    return not os.path.isdir(path)
_robust_rmtree(f'checkpoints/{tag_smoke}')
print(f'\n  PRE-FLIGHT v2 passed.')

In [ ]:
# Cell 5 -- SWEEP runner (idempotente + stream stdout + push per-run)
# Helper riusabili (definiti qui per evitare duplicazione cross-cell)
import subprocess, sys, time, os, shutil, glob, datetime
import pandas as pd

def _robust_rmtree(path, max_retries=3):
    for attempt in range(max_retries):
        if not os.path.isdir(path): return True
        shutil.rmtree(path, ignore_errors=True)
        if not os.path.isdir(path): return True
        time.sleep(0.5 * (attempt + 1))
    shutil.rmtree(path, ignore_errors=True)
    return not os.path.isdir(path)

def _build_cli(e):
    cli = [sys.executable, "train.py",
        "--training_method", "baseline",
        "--epochs", str(e["epochs"]),
        "--max_steps_per_epoch", str(e["max_steps_per_epoch"]),
        "--batch_size", "8", "--val_batch_size", "32",
        "--seq_len", str(e["seq_len"]),
        "--cf_hidden_size", str(e["hidden_size"]),
        "--cf_rank", str(e["rank"]),
        "--cf_max_delay", str(e["max_delay"]),
        "--cf_bit_shift", str(e["bit_shift"]),
        # R29 decoder fix flags (default no-op)
        "--cf_init_bias_shift", str(e.get("init_bias_shift", 0)),
        "--cf_logit_tau_init",  str(e.get("tau_init", 1.0)),
        "--cf_logit_tau_final", str(e.get("tau_final", 1.0)),
        "--cf_logit_tau_schedule", e.get("tau_schedule", "const"),
        # R25 + R30 supervisione esplicita params
        "--lambda_data", "1.0", "--lambda_phys", "0.1", "--lambda_ou", "0.05",
        "--lambda_bc", "1.0", "--lambda_sr", str(e["lambda_sr"]),
        "--lambda_T_aux", str(e["lambda_T_aux"]),
        "--lambda_v0_aux", str(e.get("lambda_v0_aux", 0.0)),
        "--lambda_s0_aux", str(e.get("lambda_s0_aux", 0.0)),
        "--lambda_a_aux",  str(e.get("lambda_a_aux", 0.0)),
        "--lambda_b_aux",  str(e.get("lambda_b_aux", 0.0)),
        "--scenario_mix", e["scenario_mix"],
        "--cut_in_ratio", str(e["cut_in_ratio"]),
        "--noise_scale", "0.0", "--po2_enabled", str(e.get("po2_enabled", 1)),
        "--n_train", "1500", "--n_val", "300",
        "--max_inf_streak", "99999", "--early_stop_patience", "0",
        "--data_cache", e["cache_path"],
        "--optimizer", e["optimizer"],
        "--lr", str(e["lr"]), "--max_lr", str(e["lr"]),
        "--scheduler", e["scheduler"],
        "--T0", str(e["T0"]),
        "--prodigy_betas", "0.9,0.99",
        "--prodigy_d_coef", str(e["d_coef"]),
        "--prodigy_d0", str(e["d0"]),
        "--prodigy_weight_decay", "0.01",
        "--prodigy_use_bias_correction", "1",
        "--prodigy_safeguard_warmup", "1",
        "--prodigy_growth_rate", str(e["growth_rate"]),
        # R30 explosion guard (default ACTIVE per i v2)
        "--max_epoch_explosion_streak", str(e.get("max_epoch_explosion_streak", 2)),
        "--epoch_explosion_threshold",  str(e.get("epoch_explosion_threshold", 100.0)),
        "--tag", e["tag"]]
    if e.get("tau_per_channel"):
        cli.extend(["--cf_logit_tau_per_channel", e["tau_per_channel"]])
    return cli

def _dst_for(e, RESULTS_DIR):
    return f"{RESULTS_DIR}/{e['axis']}/{e['tag']}"

def _push_run(e, RESULTS_DIR, BRANCH, _TMP_MSG, study_name):
    tag = e["tag"]
    src = f"checkpoints/{tag}"
    dst = _dst_for(e, RESULTS_DIR)
    if not os.path.isdir(src):
        print(f"   [WARN push] {src} mancante")
        return False
    _robust_rmtree(dst)
    os.makedirs(f"{dst}/plots", exist_ok=True)
    for f in glob.glob(f"{src}/*.csv") + glob.glob(f"{src}/*.json"):
        shutil.copy2(f, dst)
    for f in glob.glob(f"{src}/plots/*.png"):
        shutil.copy2(f, f"{dst}/plots/")
    val_str = ""
    log_path = f"{dst}/training_log.csv"
    if os.path.isfile(log_path):
        try:
            edf = pd.read_csv(log_path)
            if len(edf) > 0:
                bi = int(edf.val_total.idxmin())
                tc  = edf.get("val_T_tracking_corr", pd.Series([float("nan")])).iloc[bi]
                tci = edf.get("val_T_intra_corr",   pd.Series([float("nan")])).iloc[bi]
                val_str = (f"best val={edf.val_total.min():.4f}  T_corr={tc:.3f}  "
                           f"T_intra={tci:.3f}  (E{bi+1}/{len(edf)})")
        except Exception as ex:
            val_str = f"(log read failed: {ex})"
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    msg = (f"results ({study_name}): {tag} ({ts})\n\n{val_str}\n"
           f"desc={e.get('desc','')}\nAxis: {e['axis']}\nBranch: {BRANCH}\n")
    with open(_TMP_MSG, "w", encoding="utf-8") as fp:
        fp.write(msg)
    try:
        subprocess.run(["git","add",dst], check=True, capture_output=True)
        r = subprocess.run(["git","commit","-F",_TMP_MSG], capture_output=True, text=True)
        if r.returncode != 0:
            if "nothing to commit" in r.stdout or "nothing to commit" in r.stderr:
                return True
            print(f"   [push commit fail] {r.stderr[-300:]}"); return False
        subprocess.run(["git","pull","--no-rebase","--no-edit","origin",BRANCH],
                       capture_output=True, text=True)
        r2 = subprocess.run(["git","push","origin",BRANCH], capture_output=True, text=True)
        if r2.returncode != 0:
            print(f"   [push fail] {r2.stderr[-300:]}"); return False
        print(f"   [push OK]")
        return True
    finally:
        try: os.remove(_TMP_MSG)
        except: pass

def _run_sweep(EXPERIMENTS, RESULTS_DIR, BRANCH, _TMP_MSG, study_name, SKIP_IF_EXISTS=True):
    run_results = []
    t_start = time.time()
    total = len(EXPERIMENTS)
    for i, e in enumerate(EXPERIMENTS, 1):
        tag = e["tag"]
        dst = _dst_for(e, RESULTS_DIR)
        dst_log = f"{dst}/training_log.csv"
        if SKIP_IF_EXISTS and os.path.isfile(dst_log):
            try:
                edf = pd.read_csv(dst_log)
                v_str = f"val={edf.val_total.min():.4f} epochs={len(edf)}/{e['epochs']}"
                if len(edf) >= e["epochs"] * 0.8:
                    print(f"\n[{i}/{total}] [SKIP] {tag}: {v_str}")
                    run_results.append({"tag": tag, "axis": e["axis"], "status":"skipped"})
                    continue
            except Exception:
                pass
        print(f"\n{'='*78}\n[{i}/{total}] {tag}  [axis={e['axis']}]\n  {e.get('desc','')}\n{'='*78}")
        t0 = time.time()
        r = subprocess.run(_build_cli(e), capture_output=False)
        el = time.time() - t0
        status = "ok" if r.returncode == 0 else f"fail({r.returncode})"
        el_tot = time.time() - t_start
        done_now = sum(1 for x in run_results if x["status"]!="skipped") + 1
        eta_min = (el_tot / max(done_now,1)) * (total - i) / 60
        print(f"\n[{i}/{total}] {tag} -> {status}  ({el/60:.1f}min)  ETA={eta_min:.0f}min")
        pushed = _push_run(e, RESULTS_DIR, BRANCH, _TMP_MSG, study_name)
        run_results.append({"tag": tag, "axis": e["axis"], "status": status, "pushed": pushed})
    print(f"\n{'='*78}\nSWEEP DONE in {(time.time()-t_start)/60:.0f}min")
    return run_results

if 'EXPERIMENTS' not in dir():
    raise RuntimeError('EXPERIMENTS non definito')
run_results = _run_sweep(EXPERIMENTS, RESULTS_DIR, BRANCH, _TMP_MSG, 'R28_v2 ProdigyTuning')
print('\nSWEEP R28_v2 ProdigyTuning completato.')

In [ ]:
# Cell 6 -- Aggregator (raccoglie tutte le run + tabella sintesi)
import os, json, pandas as pd, numpy as np
from IPython.display import display, Markdown

run_dirs = []
for root, _, files in os.walk(RESULTS_DIR):
    if 'training_log.csv' in files and 'config_snapshot.json' in files:
        run_dirs.append(root)
run_dirs = sorted(run_dirs)
print(f'Run discovered: {len(run_dirs)}')

rows = []
for rd in run_dirs:
    cfg = json.load(open(os.path.join(rd, 'config_snapshot.json')))
    edf = pd.read_csv(os.path.join(rd, 'training_log.csv'))
    if len(edf) == 0: continue
    bi = int(edf['val_total'].idxmin())
    bdf = pd.read_csv(os.path.join(rd, 'training_batch_log.csv'))
    import math
    gn_f = bdf['gn_total_preclip'][bdf['gn_total_preclip'].apply(math.isfinite)]
    row = {
        'tag': cfg['tag'],
        'axis': os.path.basename(os.path.dirname(rd)),
        'n_ep': len(edf),
        'best_ep': bi+1,
        'val_total': float(edf['val_total'].iloc[bi]),
        'val_data':  float(edf['val_data'].iloc[bi]),
        'T_tracking_corr': float(edf.get('val_T_tracking_corr', pd.Series([np.nan])).iloc[bi]),
        'T_intra_corr':    float(edf.get('val_T_intra_corr',   pd.Series([np.nan])).iloc[bi]),
        'v0_pred_best':    float(edf.get('val_v0_pred_mean',   pd.Series([np.nan])).iloc[bi]),
        's0_pred_best':    float(edf.get('val_s0_pred_mean',   pd.Series([np.nan])).iloc[bi]),
        'gn_max_preclip':  float(gn_f.max()) if len(gn_f)>0 else np.nan,
        'lr': float(cfg.get('lr', np.nan)),
        'lambda_T_aux': float(cfg.get('lambda_T_aux', 0.0)),
    }
    rows.append(row)
df = pd.DataFrame(rows).sort_values('T_intra_corr', ascending=False)
df.to_csv(AGGREGATE_CSV, index=False)
print(f'Saved: {AGGREGATE_CSV}')
display(Markdown('## Tabella (ordine: T_intra_corr desc)'))
display(df.round(4))

In [ ]:
# Cell 7 -- Summary + push aggregator
import os, subprocess, tempfile, pandas as pd
if 'df' not in dir():
    df = pd.read_csv(AGGREGATE_CSV)
print(f'\n# Run analizzati: {len(df)}')
print(f'# Run con gn_max < 25 (clean): {(df.gn_max_preclip < 25).sum()}/{len(df)}')
print(f'# Run con T_intra > 0.1: {(df.T_intra_corr > 0.1).sum()}/{len(df)}')
print(f'# Run con T_intra > 0.058 (top R27 historic): {(df.T_intra_corr > 0.058).sum()}/{len(df)}')
best = df.sort_values('T_intra_corr', ascending=False).iloc[0]
print(f'\nBest T_intra: {best.tag} = {best.T_intra_corr:.3f}  (gn_max={best.gn_max_preclip:.2f})')

# Push aggregator
subprocess.run(['git','add', AGGREGATE_CSV], check=False)
msg = f'aggregator: {len(df)} run, best T_intra={best.T_intra_corr:.3f}'
with tempfile.NamedTemporaryFile('w', delete=False, suffix='.txt', encoding='utf-8') as fp:
    fp.write(msg); msg_path = fp.name
r = subprocess.run(['git','diff','--cached','--name-only'], capture_output=True, text=True)
if r.stdout.strip():
    subprocess.run(['git','commit','-F',msg_path], check=True)
    subprocess.run(['git','pull','--no-rebase','--no-edit','origin',BRANCH], check=True)
    subprocess.run(['git','push','origin',BRANCH], check=True)
    print('[OK] aggregator pushato.')
else:
    print('[SKIP] niente da pushare.')
try: os.unlink(msg_path)
except: pass